# Medical Transcription Text Classification

## Project Overview

This notebook implements a complete medical transcription text classification system. Medical transcriptions are clinical notes that need to be automatically categorised into one of four specialties.

### Dataset
- **Source**: `mtsamples.csv` — raw medical transcriptions
- **Processed**: `X.csv` → split into `train.csv` (90%) and `test.csv` (10%)
- **Classes** (from `classes.txt`): Surgery · Medical Records · Internal Medicine · Other

### Approach
The notebook is divided into two main parts:

| Part | Method | Key Files |
|------|--------|-----------|
| **Part 1** | RNN / LSTM Classifier (TensorFlow/Keras) | `rnn_model.h5`, `rnn_training_curves.png` |
| **Part 2** | Traditional ML Baselines (TF-IDF + scikit-learn) | `comparison_results.csv` |

### Auxiliary Files Used
| File | Purpose |
|------|---------|
| `clinical-stopwords.txt` | Medical-domain stop words (e.g. "patient", "denies", "history") |
| `vocab.txt` | SNOMED CT vocabulary — defines the tokenizer word index |
| `classes.txt` | Four class labels with their integer indices |


---
## Section 1 — Setup & Imports

We import all libraries upfront so that any missing dependency is caught immediately.


In [ ]:
# Standard library
import os
import re
import time
import warnings
warnings.filterwarnings('ignore')

# Data handling
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ── TensorFlow / Keras (RNN model) ──────────────────────────
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dropout, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ── Scikit-learn (baseline models + metrics) ─────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score

print("All libraries imported successfully!")
print(f"TensorFlow version : {tf.__version__}")
print(f"NumPy version      : {np.__version__}")
print(f"Pandas version     : {pd.__version__}")


---
## Section 2 — Load Auxiliary Files

Before touching the data we load the three helper files that control preprocessing and label mapping.

### Why these files matter
- **`clinical-stopwords.txt`** — removes high-frequency medical noise words that carry no discriminative signal (e.g. "patient", "denies"). Standard English stop-word lists miss these.
- **`vocab.txt`** — SNOMED CT controlled clinical terminology. We use this as our tokeniser vocabulary so that professional medical terms (e.g. "hemiarthroplasty") get stable integer indices instead of being treated as unknown.
- **`classes.txt`** — defines the mapping `line index → class label`.


In [ ]:
print("Loading auxiliary files...")

# ── 1. Clinical stop words ────────────────────────────────────
# Skip blank lines and comment lines (starting with #)
with open('clinical-stopwords.txt', 'r', encoding='utf-8') as f:
    stopwords = set(
        line.strip().lower()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    )
print(f"  Stop words loaded  : {len(stopwords):,} words")

# ── 2. SNOMED CT vocabulary ───────────────────────────────────
# Build word → integer index mapping (1-based; 0 is reserved for padding)
with open('vocab.txt', 'r', encoding='utf-8') as f:
    vocab_words = [line.strip().lower() for line in f if line.strip()]

# word_to_idx: maps each vocab word to a unique integer starting at 1
word_to_idx = {word: idx + 1 for idx, word in enumerate(vocab_words)}
VOCAB_SIZE = len(word_to_idx)

# Words not in vocab are assigned the UNK (unknown) index
UNK_IDX = VOCAB_SIZE + 1
print(f"  Vocabulary size    : {VOCAB_SIZE:,} terms")
print(f"  UNK token index    : {UNK_IDX}")

# ── 3. Class labels ───────────────────────────────────────────
# Each line is a class; the line number (0-indexed) is the integer label
with open('classes.txt', 'r', encoding='utf-8') as f:
    CLASSES = [line.strip() for line in f if line.strip()]
NUM_CLASSES = len(CLASSES)
print(f"  Classes loaded     : {CLASSES}")

# Build a reverse mapping for display: integer → class name
idx_to_class = {i: c for i, c in enumerate(CLASSES)}
print(f"  Label mapping      : {idx_to_class}")


---
## Section 3 — Load & Explore the Data

We load `train.csv` and `test.csv` directly — they are already the 90/10 split of `X.csv`, so **no re-splitting is needed**.

### CSV Schema
| Column | Description |
|--------|-------------|
| `label` | Integer class index (0 = Surgery, 1 = Medical Records, 2 = Internal Medicine, 3 = Other) |
| `description` | Short title / heading of the transcription |
| `text` | Full transcription text — this is our feature |


In [ ]:
print("Loading datasets...")

train_df = pd.read_csv('train.csv')
test_df  = pd.read_csv('test.csv')

# Fill any missing transcription text with an empty string
train_df['text'] = train_df['text'].fillna('')
test_df['text']  = test_df['text'].fillna('')

print(f"  Training samples : {len(train_df):,}")
print(f"  Test samples     : {len(test_df):,}")
print(f"\nColumn names: {train_df.columns.tolist()}")
print(f"\nSample row:")
print(train_df.iloc[0][['label', 'description']].to_string())
print(f"  text[:200]: {train_df.iloc[0]['text'][:200]}...")


In [ ]:
# ── Class distribution visualisation ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, df, title, colour in [
    (axes[0], train_df, 'Training Set Class Distribution', 'steelblue'),
    (axes[1], test_df,  'Test Set Class Distribution',     'coral'),
]:
    counts = df['label'].value_counts().sort_index()
    ax.bar([CLASSES[i] for i in counts.index], counts.values, color=colour, edgecolor='white')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Class', fontsize=11)
    ax.set_ylabel('Number of Samples', fontsize=11)
    ax.tick_params(axis='x', rotation=15)
    for bar, val in zip(ax.patches, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(val), ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=100, bbox_inches='tight')
plt.show()
print("Saved class_distribution.png")


---
## Section 4 — Data Preprocessing

The preprocessing pipeline has three steps applied to every transcription:

1. **Lowercase & clean** — convert to lower case, strip punctuation and digits.
2. **Stop word removal** — remove the clinical noise words from `clinical-stopwords.txt`.
3. **Tokenise using SNOMED vocab** — map each remaining word to its integer index from `vocab.txt`.  
   Words absent from the vocabulary are mapped to `UNK_IDX`.
4. **Pad / truncate** — all sequences are fixed to length `MAX_LEN = 300` tokens.

> **Note:** The same `clean_text` function is reused for the TF-IDF baselines in Part 2, ensuring a fair comparison.


In [ ]:
MAX_LEN = 300  # fixed sequence length — long enough for most clinical notes

def clean_text(text):
    """
    Step 1 & 2: lowercase, remove non-alphabetic characters, remove stop words.
    Returns a cleaned string (words joined by spaces).
    """
    text = str(text).lower()
    # Remove anything that isn't a letter or whitespace
    text = re.sub(r'[^a-z\s]', ' ', text)
    words = text.split()
    # Drop words that appear in the clinical stop-word list
    words = [w for w in words if w not in stopwords]
    return ' '.join(words)


def tokenise_and_pad(text_series):
    """
    Step 3 & 4: convert cleaned text to padded integer sequences.

    - word in vocab.txt  → word_to_idx[word]  (1 .. VOCAB_SIZE)
    - word not in vocab  → UNK_IDX             (VOCAB_SIZE + 1)
    - padding token      → 0                   (post-padding)
    """
    sequences = []
    for text in text_series:
        tokens = text.split()
        seq = [word_to_idx.get(w, UNK_IDX) for w in tokens]
        sequences.append(seq)
    # pad_sequences truncates sequences longer than MAX_LEN from the right
    # and pads shorter sequences with 0 on the right
    return pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')


# ── Apply preprocessing ────────────────────────────────────────
print("Cleaning text (train)...")
train_df['clean_text'] = train_df['text'].apply(clean_text)

print("Cleaning text (test)...")
test_df['clean_text'] = test_df['text'].apply(clean_text)

print("Tokenising and padding (train)...")
X_train_seq = tokenise_and_pad(train_df['clean_text'])

print("Tokenising and padding (test)...")
X_test_seq  = tokenise_and_pad(test_df['clean_text'])

y_train = train_df['label'].values
y_test  = test_df['label'].values

print(f"\nX_train shape : {X_train_seq.shape}  (samples × sequence length)")
print(f"X_test  shape : {X_test_seq.shape}")
print(f"y_train shape : {y_train.shape}")
print(f"y_test  shape : {y_test.shape}")
print(f"\nExample sequence (first 20 tokens of sample 0):")
print(X_train_seq[0, :20])


---
## Section 5 — Part 1: RNN / LSTM Classifier

### Why LSTM?
A standard RNN suffers from vanishing gradients when processing long sequences like medical reports. An **LSTM (Long Short-Term Memory)** network uses gating mechanisms to selectively remember or forget information, making it much better at capturing long-range dependencies in clinical text.

### Architecture

```
Input  →  Embedding(VOCAB_SIZE+2, 128, input_length=300)
       →  LSTM(64, return_sequences=False)
       →  Dropout(0.5)
       →  Dense(4, activation='softmax')
```

| Layer | Purpose |
|-------|---------|
| **Embedding** | Maps integer token indices to trainable 128-dimensional dense vectors |
| **LSTM** | Processes the sequence and outputs a single 64-dim context vector |
| **Dropout** | Randomly zeros 50% of units during training to reduce overfitting |
| **Dense (Softmax)** | Converts the context vector to class probabilities |


In [ ]:
EMBEDDING_DIM = 128
LSTM_UNITS    = 64
DROPOUT_RATE  = 0.5

# Build the sequential model
model = Sequential([
    # Embedding layer — input_dim = VOCAB_SIZE+2 to cover:
    #   index 0          : padding token
    #   indices 1..VOCAB  : vocabulary words
    #   index VOCAB+1    : UNK token
    Embedding(
        input_dim=VOCAB_SIZE + 2,
        output_dim=EMBEDDING_DIM,
        input_length=MAX_LEN,
        name='embedding'
    ),

    # LSTM layer — return_sequences=False means we only take the
    # final hidden state as a summary of the whole sequence
    LSTM(units=LSTM_UNITS, return_sequences=False, name='lstm'),

    # Dropout — regularization; only active during training
    Dropout(rate=DROPOUT_RATE, name='dropout'),

    # Output layer — 4 units (one per class) with softmax so outputs
    # are probabilities summing to 1
    Dense(units=NUM_CLASSES, activation='softmax', name='output')
], name='LSTM_Medical_Classifier')

# Compile
# sparse_categorical_crossentropy accepts integer labels (no one-hot encoding needed)
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


### Training

We train for **10 epochs** with a batch size of 32.  
A **10% validation split** is carved out of the training set so we can monitor overfitting during training without touching the held-out test set.


In [ ]:
EPOCHS          = 10
BATCH_SIZE      = 32
VALIDATION_SPLIT = 0.1  # 10% of training data used for validation

print("Training the LSTM model (this may take a few minutes)...")
print("-" * 60)

history = model.fit(
    X_train_seq, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    verbose=1
)

# Persist the trained model weights and architecture
model.save('rnn_model.h5')
print("\nModel saved as 'rnn_model.h5'")


### Training Curves

Plotting accuracy and loss across epochs helps us detect:
- **Overfitting** — training accuracy keeps rising while validation accuracy plateaus or falls.
- **Underfitting** — both curves are still improving at the final epoch (we could train longer).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── Accuracy ──────────────────────────────────────────────────
axes[0].plot(history.history['accuracy'],     label='Training',   color='steelblue', lw=2)
axes[0].plot(history.history['val_accuracy'], label='Validation', color='coral',     lw=2, linestyle='--')
axes[0].set_title('Training vs Validation Accuracy', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

# ── Loss ──────────────────────────────────────────────────────
axes[1].plot(history.history['loss'],     label='Training',   color='steelblue', lw=2)
axes[1].plot(history.history['val_loss'], label='Validation', color='coral',     lw=2, linestyle='--')
axes[1].set_title('Training vs Validation Loss', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig('rnn_training_curves.png', dpi=100, bbox_inches='tight')
plt.show()
print("Saved rnn_training_curves.png")


### Evaluation on the Test Set

We evaluate on the **held-out test set** (10% of the dataset that the model never saw during training or validation).  

Metrics reported:
- **Accuracy** — fraction of correctly classified samples.
- **Classification Report** — per-class precision, recall, and F1.
- **Macro F1-Score** — unweighted average F1 across all four classes (treats every class equally regardless of size).


In [ ]:
print("Evaluating LSTM model on the test set...")
print("-" * 60)

test_loss, test_acc = model.evaluate(X_test_seq, y_test, verbose=0)
print(f"Test Accuracy : {test_acc:.4f}")
print(f"Test Loss     : {test_loss:.4f}")

# Predict class probabilities, then take the argmax as the predicted class
y_pred_probs = model.predict(X_test_seq, verbose=0)
y_pred_rnn   = np.argmax(y_pred_probs, axis=1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rnn, target_names=CLASSES))

rnn_macro_f1 = f1_score(y_test, y_pred_rnn, average='macro')
print(f"Macro F1-Score: {rnn_macro_f1:.4f}")

# Store results for the final comparison table
rnn_accuracy = float(test_acc)
rnn_f1       = float(rnn_macro_f1)


---
## Section 6 — Part 2: Traditional ML Baselines

### Why compare against baselines?
RNNs are powerful sequence models but:
- They take significantly longer to train.
- They are **black boxes** — a doctor cannot easily understand *why* a note was classified as Surgery vs. Internal Medicine.
- Traditional TF-IDF + linear models often perform surprisingly well on text classification.

### TF-IDF Vectorisation
**TF-IDF** (Term Frequency–Inverse Document Frequency) converts each transcription into a sparse vector where each dimension represents a vocabulary word and the value captures how distinctive that word is for this document compared to the whole corpus.

We fit TF-IDF **only on the training set** to avoid data leakage.

### Models
| Model | Why include it? |
|-------|----------------|
| **Logistic Regression** | Strong linear baseline; highly interpretable (top coefficients = most predictive words) |
| **SVM (LinearSVC)** | Excellent on high-dimensional sparse text features; often very accurate |
| **Random Forest** | Ensemble of decision trees; captures non-linear keyword interactions |


In [ ]:
print("Fitting TF-IDF vectoriser on training data...")
print("(max_features=10,000 — keeps the 10k most informative n-grams)")

# Fit on clean training text only; transform both splits
tfidf = TfidfVectorizer(max_features=10_000)
X_train_tfidf = tfidf.fit_transform(train_df['clean_text'])
X_test_tfidf  = tfidf.transform(test_df['clean_text'])

print(f"  Train TF-IDF matrix : {X_train_tfidf.shape}")
print(f"  Test  TF-IDF matrix : {X_test_tfidf.shape}")


### Logistic Regression
A linear classifier that models the log-odds of each class as a linear combination of TF-IDF features.  
**Interpretability:** the weight of a word in a class's coefficient vector tells us how much that word pushes toward that class.


In [ ]:
print("Training Logistic Regression (max_iter=1000)...")
start = time.time()

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_tfidf, y_train)

lr_time = time.time() - start
lr_pred  = lr_model.predict(X_test_tfidf)
lr_acc   = accuracy_score(y_test, lr_pred)
lr_f1    = f1_score(y_test, lr_pred, average='macro')

print(f"  Accuracy    : {lr_acc:.4f}")
print(f"  Macro F1    : {lr_f1:.4f}")
print(f"  Training time: {lr_time:.2f} s")
print("\nClassification Report:")
print(classification_report(y_test, lr_pred, target_names=CLASSES))


### SVM (LinearSVC)
Support Vector Machine with a linear kernel. Finds the maximum-margin hyperplane separating the classes in the TF-IDF feature space.  
Very effective for sparse, high-dimensional text data.


In [ ]:
print("Training SVM — LinearSVC (max_iter=2000)...")
start = time.time()

svm_model = LinearSVC(max_iter=2000, random_state=42)
svm_model.fit(X_train_tfidf, y_train)

svm_time = time.time() - start
svm_pred = svm_model.predict(X_test_tfidf)
svm_acc  = accuracy_score(y_test, svm_pred)
svm_f1   = f1_score(y_test, svm_pred, average='macro')

print(f"  Accuracy    : {svm_acc:.4f}")
print(f"  Macro F1    : {svm_f1:.4f}")
print(f"  Training time: {svm_time:.2f} s")
print("\nClassification Report:")
print(classification_report(y_test, svm_pred, target_names=CLASSES))


### Random Forest
An ensemble of decision trees each trained on a random subset of features and samples.  
Captures non-linear interactions between medical keywords, but less interpretable than Logistic Regression.


In [ ]:
print("Training Random Forest (100 trees, n_jobs=-1 for parallel training)...")
start = time.time()

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_tfidf, y_train)

rf_time = time.time() - start
rf_pred  = rf_model.predict(X_test_tfidf)
rf_acc   = accuracy_score(y_test, rf_pred)
rf_f1    = f1_score(y_test, rf_pred, average='macro')

print(f"  Accuracy    : {rf_acc:.4f}")
print(f"  Macro F1    : {rf_f1:.4f}")
print(f"  Training time: {rf_time:.2f} s")
print("\nClassification Report:")
print(classification_report(y_test, rf_pred, target_names=CLASSES))


---
## Section 7 — Model Comparison

We now combine all results into a single comparison table.  
The **RNN (LSTM)** results come from Part 1; the baseline results were just computed above.

### Metrics explained
| Metric | What it measures |
|--------|-----------------|
| **Accuracy** | % of test samples correctly classified |
| **Macro F1** | Average F1 across all 4 classes (balanced — good for imbalanced datasets) |
| **Training Time** | Wall-clock seconds to fit the model |
| **Interpretability** | Can a clinician understand *why* the model made a decision? |


In [ ]:
# Build a summary dictionary for each model
baseline_results = [
    {
        'Algorithm'         : 'Logistic Regression',
        'Accuracy'          : lr_acc,
        'F1-Score (Macro)'  : lr_f1,
        'Training Time (s)' : f'{lr_time:.2f}s',
        'Interpretability'  : 'High'
    },
    {
        'Algorithm'         : 'SVM (LinearSVC)',
        'Accuracy'          : svm_acc,
        'F1-Score (Macro)'  : svm_f1,
        'Training Time (s)' : f'{svm_time:.2f}s',
        'Interpretability'  : 'Medium'
    },
    {
        'Algorithm'         : 'Random Forest',
        'Accuracy'          : rf_acc,
        'F1-Score (Macro)'  : rf_f1,
        'Training Time (s)' : f'{rf_time:.2f}s',
        'Interpretability'  : 'Low-Medium'
    },
]

# Prepend RNN results (from Part 1)
rnn_row = {
    'Algorithm'         : 'RNN (LSTM)',
    'Accuracy'          : rnn_accuracy,
    'F1-Score (Macro)'  : rnn_f1,
    'Training Time (s)' : 'Slow (multi-minute)',
    'Interpretability'  : 'Low (Black Box)'
}

all_results = [rnn_row] + baseline_results
comparison_df = pd.DataFrame(all_results)

# Format numeric columns for display
comparison_df['Accuracy']         = comparison_df['Accuracy'].apply(lambda x: f'{x:.4f}' if isinstance(x, float) else x)
comparison_df['F1-Score (Macro)'] = comparison_df['F1-Score (Macro)'].apply(lambda x: f'{x:.4f}' if isinstance(x, float) else x)

print("=" * 75)
print("MODEL COMPARISON TABLE")
print("=" * 75)
print(comparison_df.to_string(index=False))
print("=" * 75)

# Save to CSV for reproducibility
comparison_df.to_csv('comparison_results.csv', index=False)
print("\nSaved comparison_results.csv")


In [ ]:
# ── Bar chart comparison ──────────────────────────────────────
# Re-read numeric values for plotting
plot_data = {
    'Algorithm' : [r['Algorithm'] for r in all_results],
    'Accuracy'  : [r['Accuracy'] if isinstance(r['Accuracy'], float) else float('nan') for r in all_results],
    'Macro F1'  : [r['F1-Score (Macro)'] if isinstance(r['F1-Score (Macro)'], float) else float('nan') for r in all_results],
}
# RNN accuracy and F1 are floats in our stored variables
plot_data['Accuracy'][0]  = rnn_accuracy
plot_data['Macro F1'][0]  = rnn_f1

x   = range(len(plot_data['Algorithm']))
w   = 0.35
fig, ax = plt.subplots(figsize=(11, 5))

bars1 = ax.bar([i - w/2 for i in x], plot_data['Accuracy'], width=w,
               label='Accuracy', color='steelblue', edgecolor='white')
bars2 = ax.bar([i + w/2 for i in x], plot_data['Macro F1'],  width=w,
               label='Macro F1', color='coral',     edgecolor='white')

# Annotate each bar with its value
for bar in bars1 + bars2:
    if bar.get_height() > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(list(x))
ax.set_xticklabels(plot_data['Algorithm'], rotation=10, fontsize=11)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Comparison: Accuracy & Macro F1-Score', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("Saved model_comparison.png")


---
## Section 8 — Conclusion & Discussion

### Summary of Findings

This notebook built and compared four models for classifying medical transcriptions into four specialties: **Surgery, Medical Records, Internal Medicine, Other**.

#### RNN (LSTM) — Part 1
- Processes text **sequentially**, preserving word order and long-range context.
- Uses the **SNOMED CT vocabulary** (`vocab.txt`) and **clinical stop words** (`clinical-stopwords.txt`) for domain-specific preprocessing.
- Slower to train but capable of capturing linguistic patterns across long clinical notes.

#### Traditional ML Baselines — Part 2
- **Logistic Regression** — typically the best "simple" baseline; very fast and each word's contribution is directly readable from the model coefficients.
- **SVM (LinearSVC)** — strong at separating classes in the high-dimensional TF-IDF space.
- **Random Forest** — ensemble method; captures non-linear keyword interactions but less interpretable.

### Clinical Deployment Trade-offs

| Concern | Preferred Model |
|---------|----------------|
| Maximum accuracy | RNN (LSTM) |
| Speed / real-time use | Logistic Regression or SVM |
| Interpretability (explaining decisions to doctors) | Logistic Regression |
| Resource-constrained environment | TF-IDF + Logistic Regression |

> **Key insight**: In medical contexts, a small accuracy gain from an RNN does not always justify the loss of interpretability. A Logistic Regression trained on TF-IDF features lets clinicians inspect *which specific terms* drove a classification decision — critical for trust, auditing, and regulatory compliance.

### Output Files Produced
| File | Description |
|------|-------------|
| `rnn_model.h5` | Saved trained LSTM model |
| `rnn_training_curves.png` | Accuracy & loss curves across epochs |
| `class_distribution.png` | Class frequency bar charts |
| `model_comparison.png` | Side-by-side accuracy/F1 bar chart |
| `comparison_results.csv` | Numeric comparison table (all models) |
